# 🔬 Xử lý Dữ liệu & Phân tích Thống kê (Data Pipeline, EDA & ANOVA)
## Dự án: Tối ưu hóa Thiết kế Microrobot bằng Học máy (Tuần 1)

### Mục tiêu chính:
1. **Đọc & Chuẩn hóa dữ liệu (27 quan sát)**:
   - Do thông số $E$ ($45 - 55\text{ mm}$) lớn gấp nhiều lần các biến $A, B, C, D$ ($0.55 - 0.9\text{ mm}$), áp dụng chuẩn hóa phối hợp:
   $$\mathbf{x}_{\text{norm}} = \frac{\mathbf{x} - \mathbf{x}_{\min}}{\mathbf{x}_{\max} - \mathbf{x}_{\min}}, \quad \mathbf{y}_{\text{norm}} = \frac{\mathbf{y} - \mu_y}{\sigma_y}$$
2. **Phân tích độ nhạy (ANOVA & Sensitivity Analysis)**:
   - Tái lập bảng ANOVA bằng Python (`statsmodels`, `scipy.stats`).
   - Xác nhận mức độ đóng góp (% contribution): biến $C$ đóng góp lớn nhất vào $Stroke$ ($37.65\%$) và $SF$ ($29.35\%$), tiếp theo là $D$ ($12.47\%$) và $E$ ($7.20\%$).
3. **Deliverables Tuần 1**:
   - Module `data_loader.py`
   - Biểu đồ nhiệt ma trận tương quan (Correlation Matrix)
   - Biểu đồ Pareto tác động biến số (Pareto Chart of Effects)

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm đường dẫn gốc vào hệ thống
sys.path.append(os.path.abspath('..'))
from data_loader import load_and_preprocess_data, compute_anova_analysis, plot_correlation_matrix, plot_pareto_impact

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 11
print("Đã import thành công pipeline data_loader!")

## 1. Nhiệm vụ 1: Đọc & Chuẩn hóa Dữ liệu Thực nghiệm DoE

In [ ]:
data_dict = load_and_preprocess_data('../data/parameter_stroke_safety_factor (1).csv')
df_raw = data_dict['df_raw']
df_norm = data_dict['df_norm']
pipeline = data_dict['pipeline']

print(f"Số lượng quan sát DoE: {len(df_raw)}")
print("=== 5 DÒNG DỮ LIỆU GỐC ĐẦU TIÊN ===")
display(df_raw.head())

In [ ]:
print("=== 5 DÒNG DỮ LIỆU ĐÃ CHUẨN HÓA (x_norm: Min-Max [0, 1], y_norm: Z-Score) ===")
display(df_norm.head())

## 2. Thống kê Mô tả & Phân bố Dữ liệu

In [ ]:
stats_summary = pd.DataFrame({
    'Min': df_raw.min(),
    'Max': df_raw.max(),
    'Mean': df_raw.mean(),
    'Std': df_raw.std(),
    'Range': df_raw.max() - df_raw.min()
})
print("=== BẢNG THỐNG KÊ MÔ TẢ ĐẦY ĐỦ CÁC THÔNG SỐ ===")
display(stats_summary.round(4))

## 3. Biểu đồ Ma trận Tương quan (Correlation Matrix Heatmap)

In [ ]:
fig_corr = plot_correlation_matrix(df_raw, save_path='../reports/figures/correlation_matrix.png', show=True)

## 4. Nhiệm vụ 2: Phân tích Phương sai (ANOVA & Sensitivity Analysis)
Tái lập bảng ANOVA bằng Python (`statsmodels`, `scipy.stats`) và định lượng tỷ lệ đóng góp (% contribution) của từng biến số hình học.

In [ ]:
anova_stroke, anova_sf = compute_anova_analysis(df_raw)

print("=== BẢNG ANOVA CHO ĐỘ DỊCH CHUYỂN STROKE (µm) ===")
display(anova_stroke)

print("\n=== BẢNG ANOVA CHO HỆ SỐ AN TOÀN SAFETY FACTOR (SF) ===")
display(anova_sf)

### Xác nhận Mức độ Đóng góp (Contribution Percentage):
- **Stroke (µm)**: Biến $C$ đóng góp **$37.65\%$** ($F=19.000$, $p=0.00028$), tiếp theo là $D$ ($12.47\%$) và $E$ ($7.20\%$).
- **Safety Factor**: Biến $C$ đóng góp **$29.35\%$** ($F=10.037$, $p=0.00463$), tiếp theo là $A$ ($5.78\%$), $B$ ($1.98\%$), $E$ ($1.43\%$), và $D$ ($0.06\%$).

## 5. Biểu đồ Pareto Tác động Biến số (Pareto Chart of Effects)

In [ ]:
fig_pareto = plot_pareto_impact(anova_stroke, anova_sf, save_path='../reports/figures/pareto_impact_chart.png', show=True)

## 6. Kết luận & Đề xuất Kỹ thuật Tuần 1
1. **Yếu tố $C$ là thông số hình học nhạy cảm và quan trọng nhất**: Chiếm tới $37.65\%$ tác động tới Stroke và $29.35\%$ tới Safety Factor với mức ý nghĩa thống kê $p < 0.005$.
2. **Yếu tố $D$ và $E$ có tác động đáng kể tới Stroke**: Đóng góp lần lượt $12.47\%$ và $7.20\%$, trong khi ít ảnh hưởng trực tiếp tới hệ số an toàn.
3. **Tập dữ liệu đã chuẩn hóa phối hợp (Min-Max cho $X$ và Z-Score cho $Y$)** sẵn sàng làm đầu vào cho các mô hình học máy phi tuyến (ANN 5-11-1 và GPR) ở các bước tiếp theo.